# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`

This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading

Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
# Display core metadata via the to_json() method for notebook display
meta_json = dataset.metadata.to_json()
print(f"{meta_json['name']}: {meta_json['description']}")

## 2. Data Overview

Review available record sets, fields, columns and their `@id`s.

In [ ]:
# Find available record sets
meta = dataset.metadata.to_json()
recordset_objs = None
if 'recordSet' in meta and meta['recordSet']:
    recordset_objs = meta['recordSet']
elif 'recordSets' in meta and meta['recordSets']:
    recordset_objs = meta['recordSets']

if not recordset_objs:
    # Show fallback if not present in root metadata (could also be under distribution)
    print("No record sets found in metadata. Check the Croissant manifest for record sets.")
else:
    print(f"Available record sets: {len(recordset_objs)}")
    for rs in recordset_objs:
        print(f"- RecordSet @id: {rs['@id']}, name: {rs.get('name','')}\n  Fields/columns:")
        if 'field' in rs:
            for f in rs['field']:
                print(f"    Field @id: {f['@id']} (name: {f.get('name','')}, dataType: {f.get('dataType', '')})")
        if 'column' in rs:
            for c in rs['column']:
                print(f"    Column @id: {c['@id']} (name: {c.get('name','')}, dataType: {c.get('dataType','')})")
        print()

### Manual RecordSet Discovery

If there are no RecordSets found above (as in some Croissant datasets), try parsing from the `distribution` field. Here we do so explicitly:

In [ ]:
# Parse RecordSet IDs from distribution (fallback for this dataset)
record_set_ids = []
distribution = meta.get('distribution', [])
for dist in distribution:
    if '@id' in dist:
        record_set_ids.append(dist['@id'])
if record_set_ids:
    print("RecordSets (from distribution):")
    for rec in record_set_ids:
        print(" -", rec)
else:
    print("No RecordSet IDs located in standard fields. Please inspect the Croissant JSON for record sets.")

#### Inspect Data Schema for RecordSet details

If the exact contents or schema for the RecordSets is needed (fields and columns and their `@id`), load them explicitly for a known distribution or by peeking at a sample record.

In [ ]:
# List fields/columns for each discovered record set (using first distribution found here)
for idx, rec_id in enumerate(record_set_ids):
    print(f"\nDistribution {idx}: @id = {rec_id}")
    try:
        records_preview = list(dataset.records(record_set=rec_id))
        if records_preview:
            print(f" - Preview of fields: {list(records_preview[0].keys())}")
        else:
            print(" - No records found.")
    except Exception as e:
        print(" - Error loading records:", e)

## 3. Data Extraction

Load data from detected record set(s) into pandas DataFrames for analysis. Each record set is referenced using its `@id` as above.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

# Load all found recordsets (from record_set_ids, fallback from above)
dataframes = {}
for rec_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=rec_id))
        df = pd.DataFrame(records)
        dataframes[rec_id] = df
        print(f"Loaded DataFrame for RecordSet @id={rec_id} with {df.shape[0]} rows and columns: {list(df.columns)}\n")
    except Exception as e:
        print(f"Failed to load records for {rec_id}: {e}")

# Choose the main table (first distribution as main record set, unless otherwise documented)
main_record_set_id = record_set_ids[0] if len(record_set_ids) > 0 else None
if main_record_set_id:
    print(f"Sample head of main DataFrame ({main_record_set_id}):")
    display(dataframes[main_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)

Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, categorizing data, removing outliers, transforming data, and grouping.

In [ ]:
# We need to choose a numeric field @id for exploration. 
# We'll select 'MW [kDa]' if present, or any numeric-like field.
df = dataframes[main_record_set_id]
numeric_candidate = None
for col in df.columns:
    if col.lower().startswith('mw') or 'mass' in col.lower() or 'abundance' in col.lower():
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_candidate = col
            break
if not numeric_candidate:
    numeric_candidate = df.select_dtypes(include=['number']).columns[0] if len(df.select_dtypes(include=['number']).columns) else df.columns[0]

print(f"Using numeric field for EDA: {numeric_candidate}")
# Try to clean data for numeric field (in case strings are present)
df[numeric_candidate] = pd.to_numeric(df[numeric_candidate], errors='coerce')

threshold = df[numeric_candidate].quantile(0.9)  # Top 10% as an example
filtered_df = df[df[numeric_candidate] > threshold]
print(f"Filtered records with {numeric_candidate} > {threshold:.2f} (top 10%): {filtered_df.shape[0]} rows")
display(filtered_df.head())

# Normalize numeric values
filtered_df[f"{numeric_candidate}_normalized"] = (filtered_df[numeric_candidate] - filtered_df[numeric_candidate].mean()) / filtered_df[numeric_candidate].std()
print(f"Normalized {numeric_candidate} for filtered records:")
display(filtered_df[[numeric_candidate, f"{numeric_candidate}_normalized"]].head())

# Group by a candidate field if it exists (e.g. Protein Type, Category, Peptide type)
group_field = None
for col in df.columns:
    if any([k in col.lower() for k in ['type','category','class']]):
        group_field = col
        break
if group_field:
    grouped_df = filtered_df.groupby(group_field)[numeric_candidate].mean().to_frame()
    print(f"Grouped mean {numeric_candidate} by {group_field}:")
    display(grouped_df.head())
else:
    print('No grouping field found for further EDA.')

## 5. Visualization

Visualize the distribution of the selected numeric field and the normalized values. If grouping exists, draw a barplot by group.

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
import seaborn as sns
sns.set(style="whitegrid")

# Histogram of numeric field
plt.figure(figsize=(7,4))
sns.histplot(df[numeric_candidate].dropna(), bins=40, kde=True)
plt.title(f"Distribution of {numeric_candidate}")
plt.xlabel(numeric_candidate)
plt.show()

# Boxplot on filtered values
plt.figure(figsize=(7,3))
sns.boxplot(x=filtered_df[numeric_candidate])
plt.title(f"Filtered {numeric_candidate} (Top 10%)")
plt.show()

# If grouped
if group_field:
    mean_group = filtered_df.groupby(group_field)[numeric_candidate].mean().sort_values(ascending=False)
    plt.figure(figsize=(8,4))
    mean_group[:15].plot(kind='bar')
    plt.title(f"Mean {numeric_candidate} by {group_field} (top 15)")
    plt.ylabel(numeric_candidate)
    plt.show()

## 6. Conclusion

- This notebook demonstrated how to discover, load, and explore a Croissant-structured FAIR dataset using the `mlcroissant` library.
- We referenced all entities by their `@id` (record set and field identifiers).
- Exploratory analysis illustrated typical filtering, normalization, and grouping steps on protein-level data.
- For more details, refer to the dataset's schema documentation and Croissant manifest.
